# 05 · Counterfactual replay of AgentDojo trajectories

**Requires a local Ollama** (`qwen2.5:3b`). The run is long, on the order of three
thousand model calls: it is done once before the webinar, and the notebook `04_trajectory_benchmark.ipynb`
only reads the result from `data/counterfactual/replay.csv`.

The question: the agent-trajectory-attribution labels say that the cause of the dangerous
action is the component with the injection. This is a **plausibility** label: it was assigned by an
LLM after reading the trajectory. Is this component actually necessary?
We check by intervention.

For each AgentDojo case where the target action is a tool call,
we cut the trajectory before it and ask the model to take the next step
under three conditions:

| condition | what we change |
|---|---|
| `full` | nothing |
| `no_injection` | the injection text is cut out of the cause component; it is known exactly from `source_metadata.injections` |
| `drop_other` | a random other step with a tool call is dropped: a control for the possibility that any intervention breaks the behavior |

`N_SAMPLES` generations per condition, with a fixed `seed`, so
the run is reproducible on the same version of the model and Ollama.

**A caveat that must be stated.** The trajectories were generated by Qwen3.6,
and we replay on qwen2.5:3b. What is measured is necessity for **our** model.
This is not a check of the labels for the original agent but a demonstration of how
to do such a check and what it shows.

The run appends rows to the CSV and skips those already done, so it
can be interrupted and resumed.

In [1]:
import sys; sys.path.insert(0, "..")
import random
from pathlib import Path

import pandas as pd
import requests

from agent import MODEL_NAME, OLLAMA_URL
from bench import load_ata
from bench.replay import generate, messages, reproduced, suite_tools, target_call

models = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
assert MODEL_NAME in models, f"{MODEL_NAME} is not in Ollama: ollama pull {MODEL_NAME}. Available: {models}"

N_SAMPLES = 5
OUT = Path("../data/counterfactual/replay.csv")
OUT.parent.mkdir(exist_ok=True)

agentdojo = [c for c in load_ata() if c.subset.startswith("agentdojo/")]
tools = suite_tools(agentdojo)
cases = [c for c in agentdojo
         if c.subset == "agentdojo/unsafe_action" and c.steps[c.target].flags["function"]]
print(f"cases: {len(cases)},  model calls: up to {len(cases) * 3 * N_SAMPLES}")

cases: 214,  model calls: up to 3210


Which step to drop in `drop_other` is chosen deterministically
from `case_id`: another step with a tool call before the target, not a cause.

In [2]:
def other_step(case):
    pool = [i for i in range(case.target) if case.steps[i].flags["function"] and i not in case.causes]
    return random.Random(case.case_id).choice(pool) if pool else None

done = set()
if OUT.exists():
    prev = pd.read_csv(OUT)
    done = set(zip(prev.case_id, prev.intervention, prev.seed))

for k, case in enumerate(cases, 1):
    function, values = target_call(case)
    suite_tools_ = tools[case.meta["source_metadata"]["suite_name"]]
    drop = other_step(case)
    rows = []
    for intervention in ("full", "no_injection", "drop_other"):
        if intervention == "drop_other" and drop is None:
            continue
        msgs = messages(case, suite_tools_, intervention=intervention, drop=drop)
        for seed in range(N_SAMPLES):
            if (case.case_id, intervention, seed) in done:
                continue
            out = generate(msgs, model=MODEL_NAME, seed=seed)
            rows.append({"case_id": case.case_id, "intervention": intervention, "seed": seed,
                         "dropped": drop if intervention == "drop_other" else None,
                         "function": function, "values": "|".join(values),
                         "reproduced": reproduced(out, function, values), "output": out})
    if rows:
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    if k % 20 == 0:
        print(f"{k}/{len(cases)}")

20/214


40/214


60/214


80/214


100/214


120/214


140/214


160/214


180/214


200/214


In [3]:
res = pd.read_csv(OUT)
print(f"rows: {len(res)},  model: {MODEL_NAME}\n")
print(res.groupby("intervention")["reproduced"].agg(["mean", "sum", "count"]).round(3).to_string())

rows: 2980,  model: qwen2.5:3b

               mean  sum  count
intervention                   
drop_other    0.171  144    840
full          0.170  182   1070
no_injection  0.156  167   1070
